# Week 1 Clean up NYC mobility data

In [1]:
import duckdb

con = duckdb.connect("taxi_project")

In [2]:
con.sql("""
DROP TABLE taxi_clean;
""")

In [ ]:
# con.sql("""
# CREATE TABLE yellow_taxi_raw AS
#         SELECT *
#         FROM '../data/nyc_taxi/trips/2025/*.parquet'
# """)

In [3]:
con.sql("""
SELECT COUNT(*) FROM yellow_taxi_raw;
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      3475226 │
└──────────────┘

In [4]:
con.sql(
    """ 
    SELECT vendorID, COUNT(*)
    FROM yellow_taxi_raw
    GROUP BY vendorID
"""
)

┌──────────┬──────────────┐
│ VendorID │ count_star() │
│  int32   │    int64     │
├──────────┼──────────────┤
│        1 │       753671 │
│        2 │      2719860 │
│        6 │          489 │
│        7 │         1206 │
└──────────┴──────────────┘

In [5]:
con.sql("""
CREATE OR REPLACE TABLE taxi_clean AS
SELECT
    tpep_pickup_datetime AS pickup_datetime,
    tpep_dropoff_datetime AS dropoff_datetime,
    passenger_count,
    trip_distance,
    PULocationID AS pu_location_id,
    DOLocationID AS do_location_id,
    fare_amount,
    tip_amount
FROM yellow_taxi_raw
WHERE 
    tpep_pickup_datetime IS NOT NULL 
    AND tpep_dropoff_datetime IS NOT NULL 
    AND passenger_count IS NOT NULL 
    AND trip_distance IS NOT NULL 
    AND PULocationID IS NOT NULL 
    AND DOLocationID IS NOT NULL 
    AND fare_amount IS NOT NULL 
    AND tip_amount IS NOT NULL 
    AND tpep_dropoff_datetime > tpep_pickup_datetime
    AND passenger_count > 0 
    AND trip_distance > 0
    AND fare_amount >= 0 
    AND tip_amount >= 0;
""")

In [6]:
con.sql("""
SHOW TABLES;
""")

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ taxi_clean      │
│ yellow_taxi_raw │
│ zone_lookup     │
└─────────────────┘

In [7]:
con.sql("""
SELECT COUNT(*) FROM taxi_clean;
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      2816365 │
└──────────────┘

In [8]:
con.close()